# Costruzione dello script per ottenere i dati

In [8]:
import urllib3
import requests
import pandas as pd
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
import requests
import time
from io import StringIO
from io import BytesIO


### Dati incidenti

In [3]:
# script V2 per incidenti
current_year = time.localtime().tm_year
dfs_incidenti =[]
headers = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}


for i in range(2001,current_year): #metto current_year nel range perché il range lo esclude
    url =   "https://esploradati.istat.it/SDMXWS/rest/data/41_983?startPeriod=" + str(i) + "&endPeriod="+ str(i-1)
    response_incidenti = requests.get(url, headers=headers, verify=False)

    df = pd.read_csv(StringIO(response_incidenti.text))

    dfs_incidenti.append(df)

    time.sleep(15)

df_incidenti_raw = pd.concat(dfs_incidenti, ignore_index=True)

In [4]:
df_incidenti_raw = df_incidenti_raw[['DATAFLOW','FREQ', 'REF_AREA', 'DATA_TYPE', 'RESULT', 'TIME_PERIOD', 'OBS_VALUE']]

In [5]:
df_incidenti_raw[["REF_AREA","TIME_PERIOD"]] = df_incidenti_raw[["REF_AREA","TIME_PERIOD"]].astype(int)

In [6]:
df_incidenti_raw_pivot = df_incidenti_raw.pivot(index=['DATAFLOW', 'FREQ', 'REF_AREA', 'TIME_PERIOD'],columns='RESULT', values='OBS_VALUE').reset_index()

### Dati comuni: nome, superficie, popolazione

Inizialmente ho considerato risorse da scaricare per popolazione<br>
source dati 2001-2018: https://demo.istat.it/app/?i=RBD&l=it <br>
souce dati 2019-2025: https://demo.istat.it/app/?l=it&a=2025&i=P02

Ho poi realizzato che potevo scaricare dal'API SITUAS i dati per ogni anno ciclando.

In [7]:
current_year = time.localtime().tm_year
dfs = []

for i in range(2001, current_year, 1):
    url = 'https://situas-servizi.istat.it/publish/reportspooljson?pfun=74&pdata=31/12/'+ str(i)
    r= requests.get(url, verify=False)

    df = pd.DataFrame.from_dict(r.json()['resultset'])

    dfs.append(df)

df_comuni_raw = pd.concat(dfs, ignore_index=True)

In [8]:
df_comuni_raw = df_comuni_raw[['PRO_COM', 'COMUNE', 'AREA_KMQ', 'POP_RES', 'ANNO_POP_RES', 'COD_RIP', 'COD_REG', 'COD_UTS']]

In [9]:
df_comuni_raw["ANNO_POP_RES"] = df_comuni_raw["ANNO_POP_RES"].astype(int)

### Unire i datasets: incidenti + comuni

In [10]:
#new_df = df_ricevente.merge(df_daAttaccarci[['col1','col2']], how='lef', left_on='coldeldatasetricevente', right_on='col del nuovo dataset da matcharci')
df_raw = df_incidenti_raw_pivot.merge(df_comuni_raw, how='left', left_on=['REF_AREA', 'TIME_PERIOD'], right_on=['PRO_COM', 'ANNO_POP_RES'])

In [11]:
df_raw = df_raw[['COMUNE','REF_AREA','AREA_KMQ','POP_RES', '9', 'F', 'M','TIME_PERIOD']]
df_raw = df_raw.rename(columns={"COMUNE": "Nome Comune","REF_AREA": "Codice Comune", "AREA_KMQ": "Area(kmq)", "POP_RES": "Popolazione", "9": "Incidenti", "F": "Feriti", "M": "Morti", "TIME_PERIOD": "Anno" })

In [12]:
df_raw[['Popolazione','Incidenti', 'Feriti', 'Morti']] = df_raw[['Popolazione','Incidenti', 'Feriti', 'Morti']].round()
df_raw[['Popolazione','Incidenti', 'Feriti', 'Morti']] = df_raw[['Popolazione','Incidenti', 'Feriti', 'Morti']].astype("Int64")

In [ ]:
df_raw = df_raw[df_raw['Nome Comune'].notna()].reset_index(drop=True)

In [101]:
df_raw.to_csv('../data/incidenti_comuni.csv', index=False)

### Source per dati terrioriali: area geografica, regione

In [27]:
url_codici_unita_amministrative = 'https://www.istat.it/storage/codici-unita-amministrative/Elenco-comuni-italiani.xlsx'
response_codici_unita_amministrative = requests.get(url_codici_unita_amministrative, verify=False)
df_codici_unita_amministrative = pd.read_excel(BytesIO(response_codici_unita_amministrative.content))

In [6]:
df_codici_unita_amministrative

,Codice Regione,Codice dell'Unità territoriale sovracomunale \n(valida a fini statistici),Codice Provincia (Storico)(1),Progressivo del Comune (2),Codice Comune formato alfanumerico,Denominazione (Italiana e straniera),Denominazione in italiano,Denominazione altra lingua,Codice Ripartizione Geografica,Ripartizione geografica,...,Codice Comune numerico con 110 Province (dal 2010 al 2016),Codice Comune numerico con 107 Province (dal 2006 al 2009),Codice Comune numerico con 103 Province (dal 1995 al 2005),Codice Catastale del Comune,Codice NUTS1 2021,Codice NUTS2 2021 (3),Codice NUTS3 2021,Codice NUTS1 2024,Codice NUTS2 2024 (3),Codice NUTS3 2024
0,1,201,1,1,1001,Agliè,Agliè,NaN,1,Nord-ovest,...,1001,1001,1001,A074,ITC,ITC1,ITC11,ITC,ITC1,ITC11
1,1,201,1,2,1002,Airasca,Airasca,NaN,1,Nord-ovest,...,1002,1002,1002,A109,ITC,ITC1,ITC11,ITC,ITC1,ITC11
2,1,201,1,3,1003,Ala di Stura,Ala di Stura,NaN,1,Nord-ovest,...,1003,1003,1003,A117,ITC,ITC1,ITC11,ITC,ITC1,ITC11
3,1,201,1,4,1004,Albiano d'Ivrea,Albiano d'Ivrea,NaN,1,Nord-ovest,...,1004,1004,1004,A157,ITC,ITC1,ITC11,ITC,ITC1,ITC11
4,1,201,1,6,1006,Almese,Almese,NaN,1,Nord-ovest,...,1006,1006,1006,A218,ITC,ITC1,ITC11,ITC,ITC1,ITC11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7889,20,119,119,20,119020,Sant'Antioco,Sant'Antioco,NaN,5,Isole,...,107020,107020,92063,I294,ITG,ITG2,ITG2H,ITG,ITG2,ITG2H
7890,20,119,119,21,119021,Tratalias,Tratalias,NaN,5,Isole,...,107021,107021,92085,L337,ITG,ITG2,ITG2H,ITG,ITG2,ITG2H
7891,20,119,119,22,119022,Villamassargia,Villamassargia,NaN,5,Isole,...,107022,107022,92094,L968,ITG,ITG2,ITG2H,ITG,ITG2,ITG2H
7892,20,119,119,23,119023,Villaperuccio,Villaperuccio,NaN,5,Isole,...,107023,107023,92104,M278,ITG,ITG2,ITG2H,ITG,ITG2,ITG2H


In [ ]:
#new_df = df_ricevente.merge(df_daAttaccarci[['col1','col2']], how='lef', left_on='coldeldatasetricevente', right_on='col del nuovo dataset da matcharci')
#df_completo = df_raw.merge(df_codici_unita_amministrative[['Codice Regione', 'Denominazione Regione']], how='left', left_on='COD_REG', right_on='Codice Regione')

### Analisi righe vuote

284 codici comune del dataset inciddenti non trovano corrispettivo nel dataset dei Comuni.

Da una ispezione sono 284 singoli Comuni che appaiono un anno solo in più sulla tabella incidenti, emntre in quella dei Comuni cessano di esistere l'anno prima.

Ad esempio, il Comune 6064 registra incidenti per ogni anno fino al 2019, ma nela tabella Comuni esiste fino al 2018. Lasciando quindi una riga nella tabella incidenti senza info sul COmune.

Essendo 284 osservazioni meno del 1% del dataset, vengono tolte.

In [1]:
df_incidenti_raw_pivot
df_comuni_raw
df_raw.shape

NameError: name 'df_incidenti_raw_pivot' is not defined

In [54]:
df_raw.shape
#le righe vuote vengono dalla tabella Comuni, il codice comune in Incidenti, non ha trovato corrispettivo nella colonna codici comune Comuni. le righe quindi sono rimaste con dati incidenti ma non Comuni.

(197523, 8)

In [13]:
df_comuni_mancanti = df_raw[df_raw['Nome Comune'].isnull()]
df_comuni_mancanti['Codice Comune'].value_counts()
#ogni comune appare una sola volta, non appare per due anni di fila.

Codice Comune
6064      1
6089      1
13093     1
13109     1
13175     1
         ..
107019    1
107020    1
107021    1
107022    1
107023    1
Name: count, Length: 284, dtype: int64

In [20]:
esempio = df_comuni_mancanti.iloc[0]['Codice Comune']
df_incidenti_raw_pivot[df_incidenti_raw_pivot['REF_AREA'] == esempio]


RESULT,DATAFLOW,FREQ,REF_AREA,TIME_PERIOD,9,F,M
21695,IT1:41_983(1.0),A,6064,2001,0.0,0.0,0.0
21696,IT1:41_983(1.0),A,6064,2002,0.0,0.0,0.0
21697,IT1:41_983(1.0),A,6064,2003,0.0,0.0,0.0
21698,IT1:41_983(1.0),A,6064,2004,0.0,0.0,0.0
21699,IT1:41_983(1.0),A,6064,2005,1.0,1.0,0.0
21700,IT1:41_983(1.0),A,6064,2006,0.0,0.0,0.0
21701,IT1:41_983(1.0),A,6064,2007,0.0,0.0,0.0
21702,IT1:41_983(1.0),A,6064,2008,0.0,0.0,0.0
21703,IT1:41_983(1.0),A,6064,2009,0.0,0.0,0.0
21704,IT1:41_983(1.0),A,6064,2010,0.0,0.0,0.0


In [23]:
df_comuni_raw[df_comuni_raw['PRO_COM'] == esempio]

,PRO_COM,COMUNE,AREA_KMQ,POP_RES,ANNO_POP_RES,COD_RIP,COD_REG,COD_UTS
920,6064,Cuccaro Monferrato,5.2191,364,2001,1,01,006
9022,6064,Cuccaro Monferrato,5.2191,356,2002,1,01,006
17124,6064,Cuccaro Monferrato,5.2191,371,2003,1,01,006
25224,6064,Cuccaro Monferrato,5.2191,360,2004,1,01,006
33325,6064,Cuccaro Monferrato,5.2191,367,2005,1,01,006
41426,6064,Cuccaro Monferrato,5.2191,354,2006,1,01,006
49527,6064,Cuccaro Monferrato,5.2191,343,2007,1,01,006
57628,6064,Cuccaro Monferrato,5.2191,354,2008,1,01,006
65729,6064,Cuccaro Monferrato,5.2191,351,2009,1,01,006
73829,6064,Cuccaro Monferrato,5.2191,343,2010,1,01,006


## vediamo vediamo....

In [37]:
df_raw

,Nome Comune,Codice Comune,Area(kmq),Popolazione,Incidenti,Feriti,Morti,Anno
0,Agliè,1001,13.1462,2557,5,10,0,2001
1,Agliè,1001,13.1462,2538,5,10,0,2002
2,Agliè,1001,13.1462,2588,4,7,0,2003
3,Agliè,1001,13.1462,2679,9,13,0,2004
4,Agliè,1001,13.1462,2674,2,2,0,2005
...,...,...,...,...,...,...,...,...
197518,Villaspeciosa,111107,27.1943,2536,5,7,0,2021
197519,Villaspeciosa,111107,27.1943,2575,1,1,0,2022
197520,Villaspeciosa,111107,27.1943,2616,4,5,0,2023
197521,Villaspeciosa,111107,27.1943,2628,3,5,0,2024


In [50]:
df_codici_unita_amministrative.info()

<class 'pandas.DataFrame'>
RangeIndex: 7894 entries, 0 to 7893
Data columns (total 27 columns):
 #   Column                                                                           Non-Null Count  Dtype
---  ------                                                                           --------------  -----
 0   Codice Regione                                                                   7894 non-null   int64
 1   Codice dell'Unità territoriale sovracomunale 
(valida a fini statistici)         7894 non-null   int64
 2   Codice Provincia (Storico)(1)                                                    7894 non-null   int64
 3   Progressivo del Comune (2)                                                       7894 non-null   int64
 4   Codice Comune formato alfanumerico                                               7894 non-null   int64
 5   Denominazione (Italiana e straniera)                                             7893 non-null   str  
 6   Denominazione in italiano          

In [31]:
#new_df = df_ricevente.merge(df_daAttaccarci[['col1','col2']], how='lef', left_on='coldeldatasetricevente', right_on='col del nuovo dataset da matcharci')
unione_test = df_raw.merge(
    df_codici_unita_amministrative[[
        "Codice Comune formato alfanumerico",
        "Ripartizione geografica",
        "Denominazione Regione",
        "Denominazione dell'Unità territoriale sovracomunale \n(valida a fini statistici)",
        "Flag Comune capoluogo di Provincia/Città metropolitana/libero consorzio",
    ]],
    how="left",
    left_on="Codice Comune",
    right_on="Codice Comune formato alfanumerico",
)

In [38]:
df_0105 = df_raw[df_raw["Anno"].between(2001, 2005)]

In [40]:
unione_test = df_0105.merge(
    df_codici_unita_amministrative[[
        "Codice Comune numerico con 103 Province (dal 1995 al 2005)",
        "Ripartizione geografica",
        "Denominazione Regione",
        "Denominazione dell'Unità territoriale sovracomunale \n(valida a fini statistici)"
    ]],
    how="left",
    left_on="Codice Comune",
    right_on="Codice Comune numerico con 103 Province (dal 1995 al 2005)")

In [48]:
unione_test.isnull().sum()

Nome Comune                                                                            3
Codice Comune                                                                          0
Area(kmq)                                                                              3
Popolazione                                                                            3
Incidenti                                                                              0
Feriti                                                                                 0
Morti                                                                                  0
Anno                                                                                   0
Codice Comune numerico con 103 Province (dal 1995 al 2005)                          1676
Ripartizione geografica                                                             1676
Denominazione Regione                                                               1676
Denominazione dell'Un